# AgenticRAG-Bench — Week 2
## Proper Evaluation Harness

**What changed from Week 1:**
- Real MuSiQue supporting paragraphs as knowledge base (not fake 15-doc KB)
- `max_steps=10` hard limit — no more 27-step retrieval loops
- Loop detection — flags when agent repeats the same query
- Proper D1 and D5 metric classes
- `nomic-embed-text` for embeddings (dedicated retrieval model, better than llama3.1:8b for search)
- Runs on 5 questions instead of 3
- Everything saved with week2_ prefix

**Prerequisites:**
- `ollama serve` running in separate terminal
- `llama3.1:8b` pulled
- `nomic-embed-text` pulled: `ollama pull nomic-embed-text`
- venv active with all packages installed

---

## Cell 0 — Environment check

In [1]:
import sys
import os
from dotenv import load_dotenv
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
print("Project root:", PROJECT_ROOT)

# Sequential RAGAS evaluation — required for local Ollama
os.environ["RAGAS_DO_NOT_PARALLELIZE"] = "true"

print("Python:", sys.version)

if sys.version_info < (3, 10):
    raise RuntimeError(
        f"Python {sys.version_info.major}.{sys.version_info.minor} detected. "
        "Need 3.10+. Activate your venv: source venv/bin/activate"
    )

print("Python version OK")

import requests

# Check Ollama
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    models = [m['name'] for m in r.json().get('models', [])]
    print("Ollama running. Models:", models)

    has_llama = any('llama3.1' in m for m in models)
    has_nomic = any('nomic' in m for m in models)

    if not has_llama:
        print("WARNING: llama3.1:8b not found. Run: ollama pull llama3.1:8b")
    if not has_nomic:
        print("WARNING: nomic-embed-text not found.")
        print("Run this now in a terminal: ollama pull nomic-embed-text")
        print("It is ~270MB and takes about 1 minute.")
    if has_llama and has_nomic:
        print("Both required models are available")
except Exception as e:
    raise RuntimeError(
        "Ollama not running. Open a new terminal and run: ollama serve"
    ) from e

Project root: /Users/idhantsingh/Desktop/agenticrag-bench
Python: 3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 17.0.0 (clang-1700.6.3.2)]
Python version OK
Ollama running. Models: ['nomic-embed-text:latest', 'llama3.1:8b']
Both required models are available


In [2]:
# All imports
import json
import time
import math
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Optional
from dotenv import load_dotenv

from datasets import load_dataset
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.tools import tool
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langgraph.prebuilt import create_react_agent

load_dotenv(PROJECT_ROOT / ".env")

# Create output folders
Path(PROJECT_ROOT / "data/questions").mkdir(parents=True, exist_ok=True)
Path(PROJECT_ROOT / "data/trajectories").mkdir(parents=True, exist_ok=True)
Path(PROJECT_ROOT / "notes").mkdir(parents=True, exist_ok=True)

print("All imports OK")
print("Output folders ready")

All imports OK
Output folders ready


---
## Cell 1 — Load 10 MuSiQue questions WITH their supporting paragraphs

This is the key fix from Week 1. Instead of a fake 15-document KB, each question now brings its own relevant supporting documents from the MuSiQue dataset itself.

The `paragraphs` field contains 20 paragraphs per question — a mix of supporting (relevant) and distractor (irrelevant) paragraphs. This is realistic: the agent has to find the right ones.

In [3]:
print("Loading MuSiQue dataset...")

dataset = None

load_attempts = [
    ("google-research-datasets/MuSiQue", "musique_ans_v1.0"),
    ("google-research-datasets/musique", "musique_ans_v1.0"),
    ("dgslibisey/MuSiQue", None),
]

for name, config in load_attempts:
    try:
        print(f"Trying: {name}")
        if config:
            dataset = load_dataset(name, config)
        else:
            dataset = load_dataset(name)
        print(f"Loaded from: {name}")
        break
    except Exception as e:
        print(f"Failed: {name} → {e}")

if dataset is None:
    raise ValueError("Could not load MuSiQue dataset from any source")

# Inspect the structure of one item so you understand what we're working with
sample = dataset['validation'][0]
print("\nDataset fields:", list(sample.keys()))
print("\nSample question:", sample['question'])
print("Sample answer:", sample['answer'])
print("Number of paragraphs:", len(sample['paragraphs']))
print("\nFirst 3 paragraphs:")
for i, p in enumerate(sample['paragraphs'][:3]):
    print(f"  Para {i+1} [supporting={p['is_supporting']}]: {p['paragraph_text'][:100]}...")

supporting_count = sum(1 for p in sample['paragraphs'] if p['is_supporting'])
print(f"\nSupporting paragraphs: {supporting_count}/{len(sample['paragraphs'])}")
print("The rest are distractors — realistic noisy retrieval scenario")

Loading MuSiQue dataset...
Trying: google-research-datasets/MuSiQue
Failed: google-research-datasets/MuSiQue → Dataset 'google-research-datasets/MuSiQue' doesn't exist on the Hub or cannot be accessed.
Trying: google-research-datasets/musique
Failed: google-research-datasets/musique → Dataset 'google-research-datasets/musique' doesn't exist on the Hub or cannot be accessed.
Trying: dgslibisey/MuSiQue
Loaded from: dgslibisey/MuSiQue

Dataset fields: ['id', 'paragraphs', 'question', 'question_decomposition', 'answer', 'answer_aliases', 'answerable']

Sample question: Who is the spouse of the Green performer?
Sample answer: Miquette Giraudy
Number of paragraphs: 20

First 3 paragraphs:
  Para 1 [supporting=False]: Grant's First Stand is the debut album by American jazz guitarist Grant Green featuring performances...
  Para 2 [supporting=False]: Actress / director / singer Phylicia Rashād is the older sister of performer Debbie Allen, who is ma...
  Para 3 [supporting=False]: For the ancie

In [4]:
# Load 10 questions with their full paragraph sets
questions = []

for i, item in enumerate(dataset['validation']):
    if i >= 10:
        break

    # Separate supporting from distractor paragraphs
    supporting = [p['paragraph_text'] for p in item['paragraphs'] if p['is_supporting']]
    distractors = [p['paragraph_text'] for p in item['paragraphs'] if not p['is_supporting']]
    all_paragraphs = [p['paragraph_text'] for p in item['paragraphs']]

    questions.append({
        "id": item['id'],
        "question": item['question'],
        "answer": item['answer'],
        "num_hops": len(item['question_decomposition']),
        "decomposition": [
            {"sub_q": d['question'], "sub_a": d['answer']}
            for d in item['question_decomposition']
        ],
        "paragraphs": all_paragraphs,          # ALL paragraphs for the KB
        "supporting_paragraphs": supporting,   # Ground truth relevant docs
        "distractor_paragraphs": distractors,  # Noise docs
        "num_supporting": len(supporting),
        "num_distractors": len(distractors),
    })

# Save
with open(PROJECT_ROOT / "data/questions/2_musique_10.json", "w") as f:
    json.dump(questions, f, indent=2)

print(f"Saved {len(questions)} questions")
print()
for i, q in enumerate(questions[:5]):
    print(f"Q{i+1} [{q['num_hops']}-hop]: {q['question']}")
    print(f"   Answer: {q['answer']}")
    print(f"   KB size: {len(q['paragraphs'])} docs ({q['num_supporting']} relevant + {q['num_distractors']} distractors)")
    print()

Saved 10 questions

Q1 [2-hop]: Who is the spouse of the Green performer?
   Answer: Miquette Giraudy
   KB size: 20 docs (2 relevant + 18 distractors)

Q2 [2-hop]: Who founded the company that distributed the film UHF?
   Answer: Mike Medavoy
   KB size: 20 docs (2 relevant + 18 distractors)

Q3 [2-hop]: What administrative territorial entity is the owner of Ciudad Deportiva located?
   Answer: Tamaulipas
   KB size: 20 docs (2 relevant + 18 distractors)

Q4 [2-hop]: Where is Ulrich Walter's employer headquartered?
   Answer: Cologne
   KB size: 20 docs (2 relevant + 18 distractors)

Q5 [2-hop]: Which company owns the manufacturer of Learjet 60?
   Answer: Bombardier Inc.
   KB size: 20 docs (2 relevant + 18 distractors)



---
## Cell 2 — The TrajectoryLogger class (Week 2 version)

This is a proper class now, not just a global list. It adds:
- Loop detection (same query repeated)
- Per-step relevance scoring against supporting paragraphs
- Clean reset between questions

In [5]:
class TrajectoryLogger:
    """
    Records every retrieval step an agent makes.
    Week 2 additions: loop detection, per-step relevance scoring.
    """

    def __init__(self, max_steps: int = 10):
        self.max_steps = max_steps
        self.steps = []
        self.loop_detected = False
        self.loop_query = None

    def reset(self):
        self.steps = []
        self.loop_detected = False
        self.loop_query = None

    def log(self,
            query: str,
            retrieved_docs: list,
            latency_ms: float,
            supporting_paragraphs: list = None):
        """
        Record one retrieval step.

        Args:
            query: The search query the agent used
            retrieved_docs: List of document strings retrieved
            latency_ms: How long the retrieval took
            supporting_paragraphs: Ground truth relevant docs (for D2 scoring)
        """

        # Loop detection — same query as previous step?
        is_loop = False
        if self.steps and self.steps[-1]['query'] == query:
            is_loop = True
            self.loop_detected = True
            self.loop_query = query

        # D2: per-step retrieval relevance
        # How many of the retrieved docs are actually in the supporting set?
        precision_at_k = 0.0
        if supporting_paragraphs and retrieved_docs:
            hits = sum(
                1 for doc in retrieved_docs
                if any(sup[:50] in doc for sup in supporting_paragraphs)
            )
            precision_at_k = round(hits / len(retrieved_docs), 3)

        tokens_estimate = (len(query) + sum(len(d) for d in retrieved_docs)) // 4

        step = {
            "step_id": len(self.steps) + 1,
            "query": query,
            "num_docs_retrieved": len(retrieved_docs),
            "retrieved_content": retrieved_docs,
            "precision_at_k": precision_at_k,   # D2 per-step score
            "is_loop_step": is_loop,             # D3 planning coherence signal
            "tokens_estimate": tokens_estimate,
            "latency_ms": round(latency_ms, 1)
        }
        self.steps.append(step)
        return step

    def at_limit(self) -> bool:
        return len(self.steps) >= self.max_steps

    def summary(self) -> dict:
        if not self.steps:
            return {}
        total_steps = len(self.steps)
        unique_queries = list(set(s['query'] for s in self.steps))
        redundant = total_steps - len(unique_queries)
        total_tokens = sum(s['tokens_estimate'] for s in self.steps)
        avg_precision = sum(s['precision_at_k'] for s in self.steps) / total_steps
        loop_steps = sum(1 for s in self.steps if s['is_loop_step'])

        return {
            "total_steps": total_steps,
            "unique_queries": unique_queries,
            "redundant_queries": redundant,
            "redundancy_rate": round(redundant / max(total_steps, 1), 3),
            "loop_detected": self.loop_detected,
            "loop_steps": loop_steps,
            "avg_step_precision_at_k": round(avg_precision, 3),  # D2
            "total_tokens": total_tokens,
            "total_latency_ms": round(sum(s['latency_ms'] for s in self.steps), 1),
            "tokens_per_step": round(total_tokens / max(total_steps, 1), 1)
        }


# Test it
logger = TrajectoryLogger(max_steps=10)
logger.log("test query", ["doc1", "doc2"], 100.0, ["doc1 is relevant"])
logger.log("test query", ["doc1", "doc2"], 101.0)  # same query — loop!
s = logger.summary()
print("TrajectoryLogger test:")
print(f"  Steps logged: {s['total_steps']}")
print(f"  Loop detected: {s['loop_detected']}")
print(f"  Redundant: {s['redundant_queries']}")
print("TrajectoryLogger OK")

TrajectoryLogger test:
  Steps logged: 2
  Loop detected: True
  Redundant: 1
TrajectoryLogger OK


---
## Cell 3 — D1 and D5 metric classes

These are proper, reusable metric implementations. Week 3-6 will add D2, D3, D4, D6.

In [6]:
class D1_AnswerCorrectness:
    """
    Dimension 1: Answer correctness using three complementary signals.
    
    No hardcoded domain rules. Uses:
    - Exact substring match
    - Token recall (how much of ground truth appears in prediction)
    - SequenceMatcher ratio (handles "Bombardier Inc." vs "Bombardier Aerospace")
    """

    name = "D1_answer_correctness"

    STOPWORDS = {
        "the", "a", "an", "is", "was", "of", "in", "to", "and",
        "or", "it", "its", "be", "been", "being", "have", "has",
        "had", "by", "at", "from", "with", "that", "this", "are"
    }
    # No company suffixes — the model should handle these, not the metric

    def score(self, predicted: str, ground_truth: str) -> dict:
        import re
        from difflib import SequenceMatcher

        pred  = predicted.lower().strip()
        truth = ground_truth.lower().strip()

        # Signal 1: exact substring
        exact_match = truth in pred

        # Signal 2: token recall
        # What fraction of ground truth's meaningful words appear in prediction?
        def meaningful_words(text):
            text = re.sub(r'[^\w\s]', ' ', text)
            return {w for w in text.split() if w not in self.STOPWORDS and len(w) > 1}

        pred_words  = meaningful_words(pred)
        truth_words = meaningful_words(truth)

        if truth_words:
            recall = len(pred_words & truth_words) / len(truth_words)
        else:
            recall = 1.0 if exact_match else 0.0

        # Signal 3: sequence similarity
        # Handles "Bombardier Inc." vs "Bombardier Aerospace" — finds longest
        # common substring. SequenceMatcher ratio on the raw strings.
        seq_ratio = SequenceMatcher(None, truth, pred[:len(truth)*3]).ratio()

        # Combine into one score
        # exact_match is binary override
        # near_match fires when recall is high OR sequence ratio is high
        near_match = (recall >= 0.6) or (seq_ratio >= 0.7 and recall >= 0.4)

        if exact_match:
            final_score = 1.0
        elif near_match:
            final_score = round(0.5 + (recall * 0.5), 3)  # 0.5–1.0 range
        else:
            intersection = pred_words & truth_words
            union        = pred_words | truth_words
            jaccard      = len(intersection) / len(union) if union else 0.0
            final_score  = round(jaccard, 3)

        return {
            "exact_match":    exact_match,
            "near_match":     near_match,
            "token_recall":   round(recall, 3),
            "seq_similarity": round(seq_ratio, 3),
            "score":          final_score
        }


class D5_TrajectoryEfficiency:
    """
    Dimension 5: How efficiently did the agent use compute?

    Scores:
    - redundancy_rate: fraction of retrieval steps that were wasted
    - tokens_per_step: average token cost per retrieval call
    - efficiency_score: 0-1 score (higher = more efficient)
    - loop_penalty: extra penalty if the agent looped
    """

    name = "D5_trajectory_efficiency"
    BASELINE_TOKENS_PER_STEP = 400  # expected tokens per step for a well-behaved agent

    def score(self, trajectory_summary: dict, answer_correct: bool) -> dict:
        total_steps = trajectory_summary.get("total_steps", 0)
        redundancy_rate = trajectory_summary.get("redundancy_rate", 0)
        tokens_per_step = trajectory_summary.get("tokens_per_step", 0)
        loop_detected = trajectory_summary.get("loop_detected", False)
        total_tokens = trajectory_summary.get("total_tokens", 0)

        # Base efficiency: penalise redundancy and excess token use
        redundancy_penalty = redundancy_rate
        token_penalty = min(1.0, tokens_per_step / (self.BASELINE_TOKENS_PER_STEP * 2))
        loop_penalty = 0.3 if loop_detected else 0.0

        raw_efficiency = 1.0 - (redundancy_penalty * 0.5) - (token_penalty * 0.3) - loop_penalty
        efficiency_score = max(0.0, round(raw_efficiency, 3))

        # Tokens per correct answer — infinity if wrong
        tokens_per_correct = total_tokens if answer_correct else float('inf')

        return {
            "total_steps": total_steps,
            "redundancy_rate": redundancy_rate,
            "tokens_per_step": tokens_per_step,
            "loop_detected": loop_detected,
            "loop_penalty_applied": loop_penalty,
            "efficiency_score": efficiency_score,
            "total_tokens": total_tokens,
            "tokens_per_correct_answer": tokens_per_correct
        }


# Test both metrics
d1 = D1_AnswerCorrectness()
d5 = D5_TrajectoryEfficiency()

test_d1 = d1.score("The answer is Paris, the capital of France", "Paris")
print("D1 test (should find Paris):")
print(f"  exact_match={test_d1['exact_match']}, score={test_d1['score']}")

test_traj = {"total_steps": 5, "redundancy_rate": 0.6,
             "tokens_per_step": 300, "loop_detected": True, "total_tokens": 1500}
test_d5 = d5.score(test_traj, answer_correct=False)
print("D5 test (loopy, wrong answer):")
print(f"  efficiency_score={test_d5['efficiency_score']}, loop_detected={test_d5['loop_detected']}")

print("\nD1 and D5 metrics OK")

D1 test (should find Paris):
  exact_match=True, score=1.0
D5 test (loopy, wrong answer):
  efficiency_score=0.287, loop_detected=True

D1 and D5 metrics OK


---
## Cell 4 — Build per-question knowledge bases with nomic-embed-text

Week 2 key change: each question gets its own FAISS vector store built from its own paragraphs. We use `nomic-embed-text` — a dedicated embedding model that gives much better retrieval quality than using `llama3.1:8b` for embeddings.

Pull it first if you have not already: `ollama pull nomic-embed-text`

In [7]:
# Load questions
with open(PROJECT_ROOT / "data/questions/2_musique_10.json") as f:
    questions = json.load(f)

# Dedicated embedding model — much better for retrieval than llama3.1:8b
print("Loading nomic-embed-text embedding model...")
embedding_model = OllamaEmbeddings(model="nomic-embed-text")

# Quick test
test_embed = embedding_model.embed_query("test")
print(f"Embedding model OK — vector dimension: {len(test_embed)}")
print("nomic-embed-text produces 768-dimensional vectors")
print("(vs llama3.1:8b which produces 4096-dim vectors — slower and noisier for retrieval)")

Loading nomic-embed-text embedding model...
Embedding model OK — vector dimension: 768
nomic-embed-text produces 768-dimensional vectors
(vs llama3.1:8b which produces 4096-dim vectors — slower and noisier for retrieval)


In [8]:
# Build a per-question vector store for the first 5 questions
# Each question gets a FAISS index of its own 20 paragraphs

print("Building per-question FAISS indexes...")
print("(Each question has ~20 paragraphs — mix of relevant and distractor docs)")
print()

question_stores = {}  # question_id -> FAISS vectorstore

for q in questions[:5]:  # only first 5 for Week 2
    t0 = time.time()

    docs = [Document(page_content=p) for p in q['paragraphs']]
    store = FAISS.from_documents(docs, embedding_model)
    question_stores[q['id']] = store

    elapsed = time.time() - t0
    print(f"  Q: {q['question'][:60]}...")
    print(f"     {len(docs)} docs indexed in {elapsed:.1f}s")
    print(f"     Answer: {q['answer']}")

    # Sanity check — does a search for the answer return supporting docs?
    test_results = store.similarity_search(q['answer'], k=3)
    hits = sum(
        1 for r in test_results
        if any(sup[:40] in r.page_content for sup in q['supporting_paragraphs'])
    )
    print(f"     Sanity check: searching answer '{q['answer']}' → {hits}/3 relevant docs in top-3")
    print()

print(f"Built {len(question_stores)} per-question vector stores")

Building per-question FAISS indexes...
(Each question has ~20 paragraphs — mix of relevant and distractor docs)

  Q: Who is the spouse of the Green performer?...
     20 docs indexed in 0.4s
     Answer: Miquette Giraudy
     Sanity check: searching answer 'Miquette Giraudy' → 1/3 relevant docs in top-3

  Q: Who founded the company that distributed the film UHF?...
     20 docs indexed in 0.3s
     Answer: Mike Medavoy
     Sanity check: searching answer 'Mike Medavoy' → 0/3 relevant docs in top-3

  Q: What administrative territorial entity is the owner of Ciuda...
     20 docs indexed in 0.4s
     Answer: Tamaulipas
     Sanity check: searching answer 'Tamaulipas' → 0/3 relevant docs in top-3

  Q: Where is Ulrich Walter's employer headquartered?...
     20 docs indexed in 0.3s
     Answer: Cologne
     Sanity check: searching answer 'Cologne' → 0/3 relevant docs in top-3

  Q: Which company owns the manufacturer of Learjet 60?...
     20 docs indexed in 0.3s
     Answer: Bombardie

---
## Cell 5 — Build the agent with max_steps limit

The single most important fix from Week 1. `max_iterations=10` prevents the 27-step loop.

The retrieval tool is now aware of which question is being run, so it can use the right per-question FAISS store and log precision against the right supporting paragraphs.

In [9]:
# Global state for the current question context
# (needed because LangChain tools cannot take extra parameters directly)
current_question_context = {
    "store": None,
    "supporting_paragraphs": [],
    "logger": None
}

TRAJECTORY_LOGGER = TrajectoryLogger(max_steps=10)


@tool
def search_knowledge_base(query: str) -> str:
    """
    Search the knowledge base for information relevant to the query.
    Use this tool to find facts you need to answer the question.
    You can call this multiple times with DIFFERENT queries if needed.
    If a query returns nothing useful, try a different, more specific query.
    """
    ctx = current_question_context

    # Hard stop — enforced in the tool itself
    if TRAJECTORY_LOGGER.at_limit():
        return "[SEARCH LIMIT REACHED: max 10 searches allowed per question. Please provide your best answer now.]"

    t0 = time.time()
    results = ctx["store"].similarity_search(query, k=5)
    latency = (time.time() - t0) * 1000

    result_texts = [r.page_content for r in results]

    # Log with precision scoring
    step = TRAJECTORY_LOGGER.log(
        query=query,
        retrieved_docs=result_texts,
        latency_ms=latency,
        supporting_paragraphs=ctx["supporting_paragraphs"]
    )

    # Warn the agent if it is looping
    loop_warning = ""
    if step["is_loop_step"]:
        loop_warning = "\n\n[NOTE: You searched this exact query before. Please try a DIFFERENT query or provide your answer based on what you have found.]"

    formatted = "\n\n---\n\n".join(result_texts)
    return formatted + loop_warning


# Build the agent
llm = ChatOllama(model="llama3.1:8b", temperature=0)
tools = [search_knowledge_base]
agent = create_react_agent(llm, tools)

print("Agent built")
print("Model: llama3.1:8b (local Ollama)")
print("Max retrieval steps: 10 (hard limit)")
print("Loop detection: enabled (warns agent when repeating a query)")

Agent built
Model: llama3.1:8b (local Ollama)
Max retrieval steps: 10 (hard limit)
Loop detection: enabled (warns agent when repeating a query)


/var/folders/tl/vfsv8k5j2bv967f2r2x0ph7m0000gn/T/ipykernel_8443/2574932536.py:52: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


---
## Cell 6 — The run function with full metric scoring

In [10]:
d1_metric = D1_AnswerCorrectness()
d5_metric = D5_TrajectoryEfficiency()

SYSTEM_PROMPT = """You are a research assistant answering multi-hop questions.

BEFORE searching, decompose the question into steps:
- What intermediate fact do I need first?
- What final fact do I need second?

Then search for each fact separately.

After each search:
- Identify ALL possible candidate answers from the results
- Do NOT pick the first match blindly
- Verify which candidate actually matches the question

If multiple entities are found:
- Compare them and choose the best match

Do NOT answer after the first search.

Only answer AFTER:
- You have completed all required steps
- You have verified the final answer

Before giving the final answer, explicitly write:

Step 1 result: ...
Step 2 result: ...
Final answer: ...

Example:
Question: Who is the spouse of the Green performer?

Step 1: Search "who is the Green performer" → Steve Hillage
Step 2: Search "Steve Hillage spouse" → Miquette Giraudy

Final answer: Miquette Giraudy
"""


def run_question(q: dict) -> dict:
    """
    Run one question through the agent.
    Returns a fully scored result dict.
    """
    global current_question_context

    # Set up context for this question
    TRAJECTORY_LOGGER.reset()
    current_question_context["store"] = question_stores[q['id']]
    current_question_context["supporting_paragraphs"] = q['supporting_paragraphs']

    print(f"  Q: {q['question'][:70]}...")
    print(f"     Expected: {q['answer']}")

    t0 = time.time()
    try:
        messages = [
            ("system", SYSTEM_PROMPT),
            ("user", q["question"])
        ]

        result = agent.invoke({"messages": messages})
        predicted_answer = result["messages"][-1].content
        success = True
    except Exception as e:
        predicted_answer = f"ERROR: {str(e)}"
        success = False

    total_latency = (time.time() - t0) * 1000
    traj_summary = TRAJECTORY_LOGGER.summary()

    # Score D1 and D5
    d1_scores = d1_metric.score(predicted_answer, q["answer"])
    d5_scores = d5_metric.score(traj_summary, d1_scores["exact_match"])

    print(f"     Predicted: {predicted_answer[:80]}")
    print(f"     D1 correct: {d1_scores['exact_match']} (score={d1_scores['score']})")
    print(f"     Steps: {traj_summary.get('total_steps',0)}, "
          f"Redundant: {traj_summary.get('redundant_queries',0)}, "
          f"Loop: {traj_summary.get('loop_detected',False)}")
    print(f"     D5 efficiency: {d5_scores['efficiency_score']}")

    return {
        "question_id": q["id"],
        "question": q["question"],
        "ground_truth": q["answer"],
        "num_hops": q["num_hops"],
        "predicted_answer": predicted_answer,
        "success": success,
        "total_wall_latency_ms": round(total_latency, 1),
        "trajectory": TRAJECTORY_LOGGER.steps.copy(),
        "trajectory_summary": traj_summary,
        "D1_answer_correctness": d1_scores,
        "D5_trajectory_efficiency": d5_scores,
    }


print("Run function ready")

Run function ready


---
## Cell 7 — Run 5 questions

Each question takes 30–90 seconds. Total: roughly 5–8 minutes.

Watch the output. Compare the step counts to Week 1 — the max_steps limit should prevent the 27-step loop. The per-step precision should be higher because the KB now contains relevant documents.

In [11]:
with open(PROJECT_ROOT / "data/questions/2_musique_10.json") as f:
    questions = json.load(f)

questions_to_run = questions[:5]
all_results = []

print("Running 5 questions through the agent...")
print("max_steps=10 | loop detection ON | nomic-embed-text KB")
print("=" * 65)

for i, q in enumerate(questions_to_run):
    print(f"\nQuestion {i+1}/5:")
    result = run_question(q)
    all_results.append(result)
    print()

# Save
with open(PROJECT_ROOT / "data/trajectories/2_agent_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("=" * 65)
print("All 5 results saved to data/trajectories/2_agent_results.json")

Running 5 questions through the agent...
max_steps=10 | loop detection ON | nomic-embed-text KB

Question 1/5:
  Q: Who is the spouse of the Green performer?...
     Expected: Miquette Giraudy
     Predicted: Step 1 result: 
{"name": "search_knowledge_base", "parameters": {"query": "who i
     D1 correct: True (score=1.0)
     Steps: 0, Redundant: 0, Loop: False
     D5 efficiency: 1.0


Question 2/5:
  Q: Who founded the company that distributed the film UHF?...
     Expected: Mike Medavoy
     Predicted: Step 1 result: 
{"name": "search_knowledge_base", "parameters": {"query": "compa
     D1 correct: False (score=0.0)
     Steps: 0, Redundant: 0, Loop: False
     D5 efficiency: 1.0


Question 3/5:
  Q: What administrative territorial entity is the owner of Ciudad Deportiv...
     Expected: Tamaulipas
     Predicted: Step 1 result: The possible locations for Ciudad Deportiva are not specified in 
     D1 correct: False (score=0.0)
     Steps: 1, Redundant: 0, Loop: False
     D5 effic

---
## Cell 8 — Aggregate results and Week 1 vs Week 2 comparison

In [12]:
with open(PROJECT_ROOT / "data/trajectories/2_agent_results.json") as f:
    results = json.load(f)

print("=" * 65)
print("WEEK 2 — AGGREGATE RESULTS")
print("=" * 65)

# Per-question table
print(f"{'Q':3} {'Hops':6} {'Correct':8} {'D1':6} {'Steps':7} {'Redund':8} {'Loop':6} {'D5 Eff':8} {'Prec@k':7}")
print("-" * 65)

for i, r in enumerate(results):
    d1 = r['D1_answer_correctness']
    d5 = r['D5_trajectory_efficiency']
    ts = r['trajectory_summary']

    print(
        f"{i+1:3} "
        f"{r['num_hops']:6} "
        f"{'YES' if d1['exact_match'] else 'NO':8} "
        f"{d1['score']:6.3f} "
        f"{ts.get('total_steps',0):7} "
        f"{ts.get('redundant_queries',0):8} "
        f"{'YES' if ts.get('loop_detected') else 'NO':6} "
        f"{d5['efficiency_score']:8.3f} "
        f"{ts.get('avg_step_precision_at_k',0):7.3f}"
    )

print("-" * 65)

# Aggregates
n = len(results)
correct = sum(1 for r in results if r['D1_answer_correctness']['exact_match'])
avg_d1 = sum(r['D1_answer_correctness']['score'] for r in results) / n
avg_steps = sum(r['trajectory_summary'].get('total_steps', 0) for r in results) / n
avg_redundancy = sum(r['trajectory_summary'].get('redundancy_rate', 0) for r in results) / n
loops = sum(1 for r in results if r['trajectory_summary'].get('loop_detected', False))
avg_d5 = sum(r['D5_trajectory_efficiency']['efficiency_score'] for r in results) / n
avg_precision = sum(r['trajectory_summary'].get('avg_step_precision_at_k', 0) for r in results) / n
total_tokens = sum(r['trajectory_summary'].get('total_tokens', 0) for r in results)

print(f"\nAGGREGATES ({n} questions):")
print(f"  Answer accuracy:           {correct}/{n} = {correct/n:.0%}")
print(f"  Avg D1 score:              {avg_d1:.3f}")
print(f"  Avg retrieval steps/q:     {avg_steps:.1f}")
print(f"  Avg redundancy rate:       {avg_redundancy:.0%}")
print(f"  Questions with loops:      {loops}/{n}")
print(f"  Avg D5 efficiency:         {avg_d5:.3f}")
print(f"  Avg step precision@k (D2): {avg_precision:.3f}")
print(f"  Total tokens used:         {total_tokens}")

WEEK 2 — AGGREGATE RESULTS
Q   Hops   Correct  D1     Steps   Redund   Loop   D5 Eff   Prec@k 
-----------------------------------------------------------------
  1      2 YES       1.000       0        0 NO        1.000   0.000
  2      2 NO        0.000       0        0 NO        1.000   0.000
  3      2 NO        0.000       1        0 NO        0.725   0.000
  4      2 NO        0.000       1        0 NO        0.749   0.000
  5      2 YES       1.000       1        0 NO        0.849   0.200
-----------------------------------------------------------------

AGGREGATES (5 questions):
  Answer accuracy:           2/5 = 40%
  Avg D1 score:              0.400
  Avg retrieval steps/q:     0.6
  Avg redundancy rate:       0%
  Questions with loops:      0/5
  Avg D5 efficiency:         0.865
  Avg step precision@k (D2): 0.040
  Total tokens used:         1806


In [13]:
# Week 1 vs Week 2 comparison
# Load Week 1 results if they exist

week1_path = Path(PROJECT_ROOT / "data/trajectories/1_agent_results.json")

if week1_path.exists():
    with open(week1_path) as f:
        w1 = json.load(f)

    w1_correct = sum(1 for r in w1 if r.get('answer_correct', False))
    w1_steps = sum(r.get('total_retrieval_steps', 0) for r in w1)
    w1_redundant = sum(r.get('redundant_queries', 0) for r in w1)
    w1_tokens = sum(r.get('total_tokens_estimate', 0) for r in w1)
    w1_n = len(w1)

    print("=" * 65)
    print("WEEK 1 vs WEEK 2 COMPARISON")
    print("=" * 65)
    print(f"{'Metric':<30} {'Week 1':>12} {'Week 2':>12}")
    print("-" * 55)
    print(f"{'Questions run':<30} {w1_n:>12} {n:>12}")
    print(f"{'Correct answers':<30} {w1_correct:>11}/{w1_n} {correct:>11}/{n}")
    print(f"{'Total retrieval steps':<30} {w1_steps:>12} {sum(r['trajectory_summary'].get('total_steps',0) for r in results):>12}")
    print(f"{'Total redundant queries':<30} {w1_redundant:>12} {sum(r['trajectory_summary'].get('redundant_queries',0) for r in results):>12}")
    print(f"{'Total tokens':<30} {w1_tokens:>12} {total_tokens:>12}")
    print(f"{'Knowledge base':<30} {'15 fake docs':>12} {'Real MuSiQue':>12}")
    print(f"{'Max steps limit':<30} {'None':>12} {'10':>12}")
    print(f"{'Embedding model':<30} {'llama3.1:8b':>12} {'nomic-embed':>12}")
    print("-" * 55)
    print("\nKey improvement: max_steps=10 prevents infinite loops")
    print("Key improvement: real KB gives agent a fair chance to find answers")
else:
    print("Week 1 results not found at expected path.")
    print("Skipping comparison. Week 2 results saved successfully.")

WEEK 1 vs WEEK 2 COMPARISON
Metric                               Week 1       Week 2
-------------------------------------------------------
Questions run                             3            5
Correct answers                          0/3           2/5
Total retrieval steps                    32            3
Total redundant queries                  28            0
Total tokens                           4041         1806
Knowledge base                 15 fake docs Real MuSiQue
Max steps limit                        None           10
Embedding model                 llama3.1:8b  nomic-embed
-------------------------------------------------------

Key improvement: max_steps=10 prevents infinite loops
Key improvement: real KB gives agent a fair chance to find answers


---
## Cell 9 — RAGAS evaluation with Groq (same as Week 1 but on Week 2 results)

In [14]:
import math
from datasets import Dataset as HFDataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

groq_key = os.getenv("GROQ_API_KEY")
use_groq = bool(groq_key)

if use_groq:
    from langchain_groq import ChatGroq
    judge_llm = LangchainLLMWrapper(
        ChatGroq(model="llama-3.3-70b-versatile", temperature=0, api_key=groq_key)
    )
    print("Using Groq Llama 3.3 70B as RAGAS judge")
else:
    judge_llm = LangchainLLMWrapper(ChatOllama(model="llama3.1:8b", temperature=0))
    print("No GROQ_API_KEY found — using local Ollama (will be slow)")
    print("Add GROQ_API_KEY to .env for faster evaluation")

ragas_embeddings = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="nomic-embed-text"))

# Build RAGAS dataset from Week 2 results
with open(PROJECT_ROOT / "data/trajectories/2_agent_results.json") as f:
    results = json.load(f)

ragas_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
for r in results:
    ragas_data["question"].append(r["question"])
    ragas_data["answer"].append(r["predicted_answer"])
    ragas_data["ground_truth"].append(r["ground_truth"])
    all_ctx = []
    for step in r["trajectory"]:
        all_ctx.extend(step["retrieved_content"])
    ragas_data["contexts"].append(all_ctx if all_ctx else ["No context"])

ragas_dataset = HFDataset.from_dict(ragas_data)

print("\nRunning RAGAS on Week 2 results...")
t0 = time.time()
ragas_scores = evaluate(
    ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
    llm=judge_llm,
    embeddings=ragas_embeddings,
    raise_exceptions=False
)
elapsed = time.time() - t0

print(f"RAGAS completed in {elapsed:.1f}s")
print(ragas_scores)

def safe_float(v):
    if v is None: return None
    if isinstance(v, list):
        clean = [x for x in v if x is not None and not (isinstance(x, float) and math.isnan(x))]
        return round(sum(clean)/len(clean), 4) if clean else None
    try:
        f = float(v)
        return None if math.isnan(f) else round(f, 4)
    except: return None

ragas_out = {
    "faithfulness": safe_float(ragas_scores["faithfulness"]),
    "answer_relevancy": safe_float(ragas_scores["answer_relevancy"]),
    "context_precision": safe_float(ragas_scores["context_precision"]),
    "evaluation_time_seconds": round(elapsed, 1),
    "judge_model": "groq/llama-3.3-70b-versatile" if use_groq else "ollama/llama3.1:8b",
    "embedding_model": "ollama/nomic-embed-text"
}

with open(PROJECT_ROOT / "data/trajectories/2_ragas_scores.json", "w") as f:
    json.dump(ragas_out, f, indent=2)

print("\nSaved to data/trajectories/2_ragas_scores.json")

/var/folders/tl/vfsv8k5j2bv967f2r2x0ph7m0000gn/T/ipykernel_8443/1282011762.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
/var/folders/tl/vfsv8k5j2bv967f2r2x0ph7m0000gn/T/ipykernel_8443/1282011762.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
/var/folders/tl/vfsv8k5j2bv967f2r2x0ph7m0000gn/T/ipykernel_8443/1282011762.py:4: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Exa

Using Groq Llama 3.3 70B as RAGAS judge

Running RAGAS on Week 2 results...


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[6]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kp1ktwyqekc9rs7q0dsg3j8g` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99803, Requested 1481. Please try again in 18m29.376s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Exception raised in Job[9]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatil

RAGAS completed in 182.7s
{'faithfulness': 0.0000, 'answer_relevancy': 0.7388, 'context_precision': 0.0000}

Saved to data/trajectories/2_ragas_scores.json


---
## Cell 10 — The updated discrepancy analysis (Week 1 vs Week 2 RAGAS)

In [15]:
with open(PROJECT_ROOT / "data/trajectories/2_ragas_scores.json") as f:
    w2_ragas = json.load(f)

w1_ragas_path = Path(PROJECT_ROOT / "data/trajectories/1_ragas_scores.json")

print("=" * 65)
print("RAGAS vs AGENTICRAG-BENCH — CORE DISCREPANCY")
print("=" * 65)

if w1_ragas_path.exists():
    with open(w1_ragas_path) as f:
        w1_ragas = json.load(f)
    print(f"\nRAGAS scores comparison:")
    print(f"  {'Metric':<25} {'Week 1':>10} {'Week 2':>10}")
    print(f"  {'-'*48}")
    for metric in ['faithfulness', 'answer_relevancy', 'context_precision']:
        v1 = w1_ragas.get(metric)
        v2 = w2_ragas.get(metric)
        s1 = f"{v1:.4f}" if v1 is not None else "None"
        s2 = f"{v2:.4f}" if v2 is not None else "None"
        print(f"  {metric:<25} {s1:>10} {s2:>10}")

print(f"""
What RAGAS tells you about Week 2:
  faithfulness:      {w2_ragas['faithfulness']}
  answer_relevancy:  {w2_ragas['answer_relevancy']}
  context_precision: {w2_ragas['context_precision']}

What RAGAS STILL cannot tell you:
  - Whether the agent looped (our D3 planning coherence catches this)
  - Whether retrieval calls were individually relevant (our D2 catches this)
  - Whether token cost was proportionate to correctness (our D5 catches this)
  - How performance degrades when multiple difficulty axes combine (our D6)

AgenticRAG-Bench Week 2 scores:
  D1 answer correctness: {avg_d1:.3f}
  D5 trajectory efficiency: {avg_d5:.3f}
  Avg step precision@k (D2 preview): {avg_precision:.3f}
  Loops detected: {loops}/{n} questions

This 6-dimensional profile is what RAGAS collapses into 3 numbers.
""")

RAGAS vs AGENTICRAG-BENCH — CORE DISCREPANCY

RAGAS scores comparison:
  Metric                        Week 1     Week 2
  ------------------------------------------------
  faithfulness                  0.0000     0.0000
  answer_relevancy              0.9269     0.7388
  context_precision             0.0000     0.0000

What RAGAS tells you about Week 2:
  faithfulness:      0.0
  answer_relevancy:  0.7388
  context_precision: 0.0

What RAGAS STILL cannot tell you:
  - Whether the agent looped (our D3 planning coherence catches this)
  - Whether retrieval calls were individually relevant (our D2 catches this)
  - Whether token cost was proportionate to correctness (our D5 catches this)
  - How performance degrades when multiple difficulty axes combine (our D6)

AgenticRAG-Bench Week 2 scores:
  D1 answer correctness: 0.400
  D5 trajectory efficiency: 0.865
  Avg step precision@k (D2 preview): 0.040
  Loops detected: 0/5 questions

This 6-dimensional profile is what RAGAS collapses int

---
## Cell 11 — Save Week 2 summary and notes

In [16]:
with open(PROJECT_ROOT / "data/trajectories/2_agent_results.json") as f:
    results = json.load(f)
with open(PROJECT_ROOT / "data/trajectories/2_ragas_scores.json") as f:
    w2_ragas = json.load(f)

n = len(results)
correct = sum(1 for r in results if r['D1_answer_correctness']['exact_match'])
avg_d1 = sum(r['D1_answer_correctness']['score'] for r in results) / n
avg_d5 = sum(r['D5_trajectory_efficiency']['efficiency_score'] for r in results) / n
avg_steps = sum(r['trajectory_summary'].get('total_steps', 0) for r in results) / n
avg_precision = sum(r['trajectory_summary'].get('avg_step_precision_at_k', 0) for r in results) / n
loops = sum(1 for r in results if r['trajectory_summary'].get('loop_detected', False))
total_tokens = sum(r['trajectory_summary'].get('total_tokens', 0) for r in results)

notes = f"""# Week 2 Observations — AgenticRAG-Bench
Date: {time.strftime('%Y-%m-%d')}

## Changes from Week 1
- Real MuSiQue supporting paragraphs as knowledge base
- max_steps=10 hard limit on agent
- Loop detection with agent warning
- nomic-embed-text for embeddings (dedicated retrieval model)
- D1 and D5 implemented as proper metric classes
- 5 questions instead of 3

## Results
- Questions: {n}
- Correct answers: {correct}/{n} = {correct/n:.0%}
- Avg D1 score: {avg_d1:.3f}
- Avg retrieval steps: {avg_steps:.1f}
- Loops detected: {loops}/{n}
- Avg D5 efficiency: {avg_d5:.3f}
- Avg step precision@k: {avg_precision:.3f}
- Total tokens: {total_tokens}

## RAGAS scores
- faithfulness: {w2_ragas['faithfulness']}
- answer_relevancy: {w2_ragas['answer_relevancy']}
- context_precision: {w2_ragas['context_precision']}

## Key observations
[FILL IN: did correctness improve with real KB?]
[FILL IN: did the max_steps limit prevent loops?]
[FILL IN: what did step precision@k reveal about retrieval quality?]
[FILL IN: did RAGAS scores change meaningfully between Week 1 and Week 2?]

## What to build in Week 3
- Dataset construction: 400+ questions with difficulty labels
- Implement D2 (retrieval step quality) as a full metric class
- Implement D3 (planning coherence) as a full metric class
- Start noise injection for D4
"""

with open(PROJECT_ROOT / "notes/2_observations.md", "w") as f:
    f.write(notes)

print("Saved notes/2_observations.md")
print()
print("=" * 65)
print("WEEK 2 COMPLETE")
print("=" * 65)
print()
print("Files created this week:")
print("  data/questions/2_musique_10.json")
print("  data/trajectories/2_agent_results.json")
print("  data/trajectories/2_ragas_scores.json")
print("  notes/2_observations.md")

Saved notes/2_observations.md

WEEK 2 COMPLETE

Files created this week:
  data/questions/2_musique_10.json
  data/trajectories/2_agent_results.json
  data/trajectories/2_ragas_scores.json
  notes/2_observations.md
